# Kvantu lineāro sistēmu atrisināšana izmantojot nehibrīda risinajumus

Kvantu lineāro sistēmu atrisināšanas galvenais uzdevums ir atrisināt
$
Ax=b,
$
kur $A$ ir matrica un $b$ ir zināmais vektors. Klasiskajā gadījumā mēs gribam atrast pašu vektoru $x$. Kvantu gadījumā mērķis parasti ir sagatavot kvantu stāvokli
$|x\rangle$ , kas ir proporcionāls $ A^{-1}|b\rangle.$

Tas nozīmē, ka kvantu algoritms parasti neatgriež visu risinājumu klasiskā formā. Tā vietā tas sagatavo stāvokli, no kura var iegūt informāciju par risinājumu, piemēram, novērtēt kādu lielumu.

Svarīgākie parametri lineāro sistēmu atrisnāšanai izmantojot kvantu algoritmus ir:

$
N -
$
 sistēmas izmērs jeb matricas dimensija. Ja matrica ir $N \times N$, tad sistēmā ir $N$ vienādojumi un $N$ nezināmie.

$
s -
$
 matricas retums. Tas raksturo, cik maz nenulles elementu ir matricā. Matrica ir $s$-reta, ja katrā rindā vai kolonnā ir ne vairāk kā $s$ nenulles elementi.

$
\kappa -
$
 nosacītības skaitlis. Tas parāda, cik “grūta” vai nestabila ir lineārā sistēma. Ja $\kappa$ ir liels, tad mazas kļūdas datos var izraisīt lielākas kļūdas risinājumā, un sistēmu ir grūtāk atrisināt.

$
\varepsilon -
$
pieļaujamā kļūda jeb vēlamā precizitāte. Jo mazāks ir $\varepsilon$, jo precīzāku rezultātu gribam iegūt, bet parasti tas nozīmē arī lielākas skaitļošanas izmaksas.

Kvantu algoritmi šai problēmai ir interesanti tāpēc, ka dažos gadījumos to sarežģītība ir atkarīga no $N$ tikai O(poly(logN)). Taču tas ir izdevīgi tikai tad, ja matricai ir piemērota struktūra un nav nepieciešams nolasīt visu risinājuma vektoru klasiskā formā.


## HHL

HHL ir slavenākais kvantu lineāro sistēmu algoritms. Tā galvenā ideja ir izmantot matricas īpašvērtības.  

- $A$ ir $N \times N$ matrica, kas apraksta lineāro sistēmu,
- $|u_j\rangle$ ir matricas $A$ īpašvektori (ortonormēta bāze),
- $\lambda_j$ ir atbilstošās īpašvērtības,
- $|b\rangle$ ir normalizēts kvantu stāvoklis, kas kodē labās puses vektoru $b$,
- $\beta_j = \langle u_j | b \rangle$ ir projekcijas koeficienti (kompleksi skaitļi), kas nosaka, cik lielā mērā $|b\rangle$ satur katru īpašvektoru.

$$
A|u_j\rangle=\lambda_j|u_j\rangle,
$$

$$
|b\rangle=\sum_j \beta_j |u_j\rangle,
$$
tad $|b\rangle$ ir izteikts īpašvektoru bāzē, un $\beta_j$ ir šīs sadalīšanas koeficienti.

$$
A^{-1}|b\rangle=\sum_j \beta_j \frac{1}{\lambda_j}|u_j\rangle.
$$
Tas nozīmē, ka matricas inversija šajā bāzē darbojas, katru komponenti $\beta_j$ sareizinot ar $\frac{1}{\lambda_j}$, pie nosacījuma, ka $\lambda_j \neq 0$. Rezultātā tiek iegūts risinājuma vektors kvantu stāvokļa formā.

HHL mēģina amplitūdās ieviest faktoru
$
1/\lambda_j.
$

Galvenie HHL soļi ir:
- sagatavot stāvokli $|b\rangle$,
- izmantot kvantu fāzes novērtēšanu,
- iegūt īpašvērtības $\lambda_j$,
- ar kontrolētām rotācijām ieviest faktoru $1/\lambda_j$,
- iegūt stāvokli, kas ir proporcionāls $A^{-1}|b\rangle$.

HHL bija pirmais lielais algoritms šai problēmai. Tomēr tas nevienmēr ir efektīvākais variants, jo tā sarežģītība ir samērā jutīga pret nosacītības skaitli $\kappa$ un kļūdu $\varepsilon$.

$
O(s^2\kappa^2\log N/\varepsilon).
$

Jaunākajās Qiskit versijās HHL algoritma augsta līmeņa implementācija vairs nav pieejama, tādēļ tas jārealizē zemā līmenī.

## Zema līmeņa HHL

In [4]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate, RYGate, UnitaryGate
from qiskit.quantum_info import Statevector

# Klasiskais risinājums
A = np.array([
    [1, 0],
    [0, 2]
], dtype=float)

b = np.array([
    1/np.sqrt(2),
    1/np.sqrt(2)
], dtype=float)

x_classical = np.linalg.solve(A, b) # atrisina Ax=b klasiskā veidā

# HHL beigās dod normalizētu kvantu stāvokli, tāpēc arī klasisko risinājumu normalizējam
x_classical_norm = x_classical / np.linalg.norm(x_classical)

print("=== Klasiskais risinājums ===")
print("x =", np.round(x_classical, 6))
print("Normalizēts |x> =", np.round(x_classical_norm, 6))

# Matrica ir diagonāla: A = diag(1, 2), tātad
# |0> ir eigenstāvoklis ar lambda = 1
# |1> ir eigenstāvoklis ar lambda = 2

t = np.pi / 2

# U = exp(iAt)
# Tā kā A ir diagonāla, arī U ir diagonāla
U = np.diag([
    np.exp(1j * 1 * t),   # priekš lambda = 1
    np.exp(1j * 2 * t)    # priekš lambda = 2
])

# U^2 
U2 = np.linalg.matrix_power(U, 2)

# Pārveidojam matricas par Qiskit gate objektiem
U_gate = UnitaryGate(U, label="U")
U2_gate = UnitaryGate(U2, label="U^2")

# q0 = sistēmas kubits - tajā ir |b> un beigās risinājuma stāvoklis |x>
# q1 = fāzes reģistrs LSB
# q2 = fāzes reģistrs MSB
# q3 = palīgkubits - šis HHL algoritmā tiek izmantots, lai amplitūdās ieviestu faktoru 1/lambda

qc = QuantumCircuit(4)

#|b>
qc.h(0)

# priekš QPE sākumā phase reģistru ieliek superpozīcijā.
qc.h(1)
qc.h(2)

qc.append(U_gate.control(1), [1, 0])
qc.append(U2_gate.control(1), [2, 0])

# inverse QFT uz phase register
qc.append(QFTGate(2).inverse(), [1, 2])

# HHL grib panākt, lai ancillas amplitūda uz |1> būtu proporcionāla C / lambda, kur C ir konstante. Te izvēlamies C = 1.
# ja lambda = 1, tad amplitūdai uz ancilla |1> jābūt 1
# ja lambda = 2, tad 1/2

# Jāatrod attiecīgais Ry leņķis.
# Ry(theta)|0> = cos(theta/2)|0> + sin(theta/2)|1>

# Ja gribam amplitūdu alpha uz |1>, tad vajag sin(theta/2) = alpha
# lambda = 1, alpha = 1, theta = pi
# lambda = 2, alpha = 1/2, theta = pi/3

theta_l1 = np.pi
theta_l2 = np.pi / 3

# ry_l1 aktivizējas tikai tad, ja phase register = |01>
# ry_l2 aktivizējas tikai tad, ja phase register = |10>

# ja QPE atrada lambda=1, pieliekam theta=pi
# ja QPE atrada lambda=2, pieliekam theta=pi/3

ry_l1 = RYGate(theta_l1).control(2, ctrl_state='01')
ry_l2 = RYGate(theta_l2).control(2, ctrl_state='10')

# Kontroles kubiti ir q1 un q2, targets ir q3
qc.append(ry_l1, [1, 2, 3])
qc.append(ry_l2, [1, 2, 3])

# Inverse QPE
qc.append(QFTGate(2), [1, 2])
qc.append(U2_gate.control(1).inverse(), [2, 0])
qc.append(U_gate.control(1).inverse(), [1, 0])
qc.h(2)
qc.h(1)

# Statevector satur visu 4 kubitu gala stāvokli.
# Bet mums nevajag lasīt visus 16 amplitūdu elementus.
# Mums interesē tikai "veiksmīgais zars".

sv = Statevector.from_instruction(qc)
vec = sv.data

# mums vajag lai palīgkubits = 1, jeb q3 = 1 un phase register = 00, jeb q2 = 0, q1 = 0
# Tad sistēmas kubits q0 var būt 0 vai 1

# Tātad divi vajadzīgie bāzes stāvokļi ir: 
# |1 0 0 0>, index 8
# |1 0 0 1>, index 9

# Šie divi skaitļi kopā dod sistēmas stāvokli veiksmīgajā HHL zarā.

solution_branch = np.array([vec[8], vec[9]], dtype=complex)

# Šī ir varbūtība, ka ancilla = 1 un phase register = 00, jeb HHL veiksmīgais iznākums.

success_prob = np.sum(np.abs(solution_branch)**2)

# HHL dod zaru, kas ir proporcionāls risinājumam.
x_hhl = solution_branch / np.linalg.norm(solution_branch) # Lai salīdzinātu ar klasisko |x>, to normalizējam.

# Dažreiz paliek ļoti mazas imaginārās daļas, kas rodas tikai no numeriskām kļūdām.
x_hhl_real = np.real_if_close(x_hhl, tol=1000)

print("\n=== HHL rezultāts ===")
print("Veiksmīgā zara varbūtība =", round(float(success_prob), 6))
print("HHL iegūtais |x> =", np.round(x_hhl_real, 6))

print("\n=== Salīdzinājums ar klasisko atbildi ===")
print("Klasiskais |x> =", np.round(x_classical_norm, 6))
print("HHL        |x> =", np.round(x_hhl_real, 6))

difference = np.linalg.norm(x_classical_norm - x_hhl_real)
print("Atšķirība =", difference)

if difference < 1e-10:
    print("\nSecinājums: HHL rezultāts sakrīt ar klasisko risinājumu.")
else:
    print("\nSecinājums: rezultāti nesakrīt pilnīgi, jāpārbauda ķēde.")

print("\n===  HHL ķēde ===")
print(qc.draw())

=== Klasiskais risinājums ===
x = [0.707107 0.353553]
Normalizēts |x> = [0.894427 0.447214]

=== HHL rezultāts ===
Veiksmīgā zara varbūtība = 0.625
HHL iegūtais |x> = [0.894427 0.447214]

=== Salīdzinājums ar klasisko atbildi ===
Klasiskais |x> = [0.894427 0.447214]
HHL        |x> = [0.894427 0.447214]
Atšķirība = 2.7755575615628914e-16

Secinājums: HHL rezultāts sakrīt ar klasisko risinājumu.

===  HHL ķēde ===
     ┌───┐┌───┐┌─────┐                                       ┌─────────┐»
q_0: ┤ H ├┤ U ├┤ U^2 ├───────────────────────────────────────┤ Unitary ├»
     ├───┤└─┬─┘└──┬──┘┌─────────┐                    ┌──────┐└────┬────┘»
q_1: ┤ H ├──■─────┼───┤0        ├────■─────────o─────┤0     ├─────┼─────»
     ├───┤        │   │  qft_dg │    │         │     │  Qft │     │     »
q_2: ┤ H ├────────■───┤1        ├────o─────────■─────┤1     ├─────■─────»
     └───┘            └─────────┘┌───┴───┐┌────┴────┐└──────┘           »
q_3: ────────────────────────────┤ Ry(π) ├┤ Ry(π/3) ├─────────────

- - -
## Block-encoding un QSVT

Šī ir modernāka ne-hibrīda pieeja. Tās ideja ir matricu $A$ iegult lielākā unitārā operatorā un pēc tam realizēt funkciju, kas uz īpašvērtībām darbojas līdzīgi
$$
1/\lambda.
$$

### Block-encoding ideja

Tā vietā, lai tieši realizētu $A$, tiek konstruēts unitārs operators $U_A$, kas satur normalizētu matricu $A/\alpha$ kā bloku:
$$
\langle 0| U_A |0\rangle = \frac{A}{\alpha},
$$
kur $\alpha \ge \|A\|$.

Tas nozīmē, ka, ja sistēma atrodas stāvoklī $|0\rangle \otimes |b\rangle$, tad pēc $U_A$ pielietošanas daļa amplitūdas satur informāciju par
$$
\frac{A}{\alpha}|b\rangle.
$$

Ja
$$
A|u_j\rangle = \lambda_j |u_j\rangle,
$$
tad block-encoding gadījumā tiek efektīvi manipulēts ar normalizētajām vērtībām
$$
\tilde{\lambda}_j = \lambda_j / \alpha.
$$

### QSVT (Quantum Singular Value Transformation)

QSVT ļauj realizēt funkciju $f(x)$ uz matricas īpašvērtībām.

Ja
$$
|b\rangle = \sum_j \beta_j |u_j\rangle,
$$
tad pēc QSVT transformācijas iegūst:
$$
f(A/\alpha)|b\rangle = \sum_j \beta_j f(\lambda_j/\alpha) |u_j\rangle.
$$

Lai atrisinātu lineāro sistēmu, izvēlas funkciju
$$
f(x) \approx \frac{1}{x},
$$
tādējādi:
$$
f(A/\alpha)|b\rangle \approx \sum_j \beta_j \frac{\alpha}{\lambda_j} |u_j\rangle.
$$

Tātad:
$$
|x\rangle \propto A^{-1}|b\rangle.
$$

### Sarežģītība:
$$
O(s \kappa\, \mathrm{polylog}(N,\kappa,1/\varepsilon)).
$$

In [5]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import RYGate
from qiskit.quantum_info import Statevector, Operator

# Lineārā sistēma A x = b
A = np.array([
    [1, 0],
    [0, 2]
], dtype=float)

b = np.array([
    1 / np.sqrt(2),
    1 / np.sqrt(2)
], dtype=float)

# Izvēlamies alpha = 2, jo tas ir matricas A lielākais īpašvērtības lielums,
# ||A|| = 2. Lai block-encoding būtu korekts, jāizpildās ||A/alpha|| ≤ 1.
alpha = 2.0

# Šai diagonālajai matricai īpašvērtības ir diagonāles elementi
eigvals = np.diag(A)

# Klasiskais risinājums
x_classical = np.linalg.solve(A, b)
x_classical_norm = x_classical / np.linalg.norm(x_classical)  # normalizējam rezultātu

print("=== Klasiskais risinājums ===")
print("x =", np.round(x_classical, 6))
print("Normalizēts |x> =", np.round(x_classical_norm, 6))

# q0 = palīgkubits
# q1 = sistēmas kubits

# Gribam, lai <0|U_A|0> = A/alpha
# Tāpēc katrai īpašvērtībai λ_j atbilstošais koeficients ir λ_j / alpha
be_coef = eigvals / alpha

# Pie |0> stāvokļa vajag koeficientu λ_0/alpha = 1/2
# RY(theta)|0> dod amplitūdu uz |0> vienādu ar cos(theta/2),
# tāpēc leņķi iegūstam no cos(theta/2) = λ_0/alpha
theta_be = 2 * np.arccos(be_coef[0])

U_A_circuit = QuantumCircuit(2, name="U_A")  # ķēde U_A reprezentēšanai
c0_ry = RYGate(theta_be).control(1, ctrl_state='0')  # aktivizējas, ja q1 ir 0
U_A_circuit.append(c0_ry, [1, 0])

print("\n=== Block-encoding ķēde U_A ===")
print(U_A_circuit.draw())

U_A = Operator(U_A_circuit).data  # Pārvērš kvantu ķēdi par tās matricas formu
print("U_A matrica =")
print(np.round(U_A, 6))

print("\nAugšējais kreisais bloks attiecībā pret ancilla=0 zaru dod A/alpha.")
print("A/alpha =")
print(np.round(A / alpha, 6))

# Inversijas gadījumā vajag koeficientus 1/λ_j
inv_coef = 1 / eigvals

# Pie |1> stāvokļa vajag koeficientu 1/λ_1 = 1/2
# Atkal izmantojam cos(theta/2) = vajadzīgais koeficients
theta_inv = 2 * np.arccos(inv_coef[1])

U_inv_circuit = QuantumCircuit(2, name="U_inv")
c1_ry = RYGate(theta_inv).control(1, ctrl_state='1')
U_inv_circuit.append(c1_ry, [1, 0])

print("\n=== 'QSVT inverse' ķēde šim piemēram ===")
print(U_inv_circuit.draw())

U_inv = Operator(U_inv_circuit).data
print("U_inv matrica =")
print(np.round(U_inv, 6))

qc = QuantumCircuit(2)
qc.h(1)  # Sagatavo |b> uz sistēmas kubita

# Pielietojam transformēto operatoru
qc.compose(U_inv_circuit, inplace=True)

print("\n=== Gala ķēde ===")
print(qc.draw())

sv = Statevector.from_instruction(qc)  # aprēķina ķēdes gala kvantu stāvokli
vec = sv.data  # paņem gala stāvokļa amplitūdas kā NumPy masīvu

# Mūs interesē tikai tie gadījumi, kur ancilla q0 = 0,
# tāpēc ņemam amplitūdas no stāvokļiem |00> un |10>
solution_branch = np.array([vec[0], vec[2]], dtype=complex)

success_prob = float(np.sum(np.abs(solution_branch) ** 2))
x_qsvt = solution_branch / np.linalg.norm(solution_branch)
x_qsvt = np.real_if_close(x_qsvt, tol=1000)

print("\n=== 5) Rezultāts ===")
print("Veiksmīgā zara varbūtība =", round(success_prob, 6))
print("Block-encoding/QSVT iegūtais |x> =", np.round(x_qsvt, 6))

print("\n=== 6) Salīdzinājums ar klasisko atbildi ===")
print("Klasiskais |x> =", np.round(x_classical_norm, 6))
print("Kvantu     |x> =", np.round(x_qsvt, 6))

difference = np.linalg.norm(x_classical_norm - x_qsvt)
print("Atšķirība =", difference)

if difference < 1e-10:
    print("\nSecinājums: rezultāts sakrīt ar klasisko normalizēto risinājumu.")
else:
    print("\nSecinājums: rezultāti pilnīgi nesakrīt.")

=== Klasiskais risinājums ===
x = [0.707107 0.353553]
Normalizēts |x> = [0.894427 0.447214]

=== Block-encoding ķēde U_A ===
     ┌──────────┐
q_0: ┤ Ry(2π/3) ├
     └────┬─────┘
q_1: ─────o──────
                 
U_A matrica =
[[ 0.5     +0.j -0.866025+0.j  0.      +0.j  0.      +0.j]
 [ 0.866025+0.j  0.5     +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  1.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.      +0.j  1.      +0.j]]

Augšējais kreisais bloks attiecībā pret ancilla=0 zaru dod A/alpha.
A/alpha =
[[0.5 0. ]
 [0.  1. ]]

=== 'QSVT inverse' ķēde šim piemēram ===
     ┌──────────┐
q_0: ┤ Ry(2π/3) ├
     └────┬─────┘
q_1: ─────■──────
                 
U_inv matrica =
[[ 1.      +0.j  0.      +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  1.      +0.j  0.      +0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.5     +0.j -0.866025+0.j]
 [ 0.      +0.j  0.      +0.j  0.866025+0.j  0.5     +0.j]]

=== Gala ķēde ===
          ┌──────────┐
q_0: 

## Citas kvantu lineāro sistēmu risināšanas metodes

Papildus HHL un block-encoding/QSVT pieejām literatūrā tiek minētas arī šādas metodes:

- Adiabatiskās kvantu lineāro sistēmu metodes
- Diskrētās adiabatiskās metodes
- LCU (Linear Combination of Unitaries) metodes
- Čebiševa polinomu metodes
- Krylov apakštelpas metodes
- Variacionālie kvantu lineārie risinātāji (VQLS)
- Hibrīdās kvantu-klasiskās metodes
- Shadow quantum linear solvers
- Neironu tīklu / variacionālās neironu pieejas

Plašāku informāciju par šīm metodēm var atrast atsauču materiālos.

## Atsauces

Šis materiāls balstīts uz:  
Emergent Mind, *Quantum Linear Solvers (QLSS).*  
https://www.emergentmind.com/topics/quantum-linear-solvers-qlss

Emergent Mind, *Quantum Singular Value Transformation (QSVT).*  
https://www.emergentmind.com/topics/quantum-singular-value-transformation-qsvt